# 52. 조회 오류 수정 후 네 검색 방식 재비교
기존 52번은 예외 문구에 대상 단어가 있기만 해도 혜택 전체를 제외하는 오류가 확인되어 답변 API 실행 전에 중단했다. 기존 코드·239개 검색 결과·사용량은 보존한다. 이 노트북은 같은 51번 첫 번째 조건 추출을 사용하고 SQL 사실 판정만 별도 모듈로 수정한다. 원본 DB와 운영 코드는 변경하지 않는다.

S_ONLY / U / SEQ / PAR를 비교한다. U의 동일 입력 검색은 검증된 기존 결과를 재사용하며 최초 시간은 과거 실측으로 구분한다. 모든 답변은 방식별 새 요청이다. 공유 모듈은 재개 시 입력·원본 보존·비용 상한을 동일하게 검사하기 위해 필요하다. HTML은 결과를 읽은 뒤 직접 작성한다. 첫 글자 표시 시간과 화면 렌더링 시간은 측정하지 않는다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,os,time,copy,resource,shutil
import numpy as np
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('corrected_modes52',ROOT/'notebooks/scripts/52_retrieval_modes.py')
M=importlib.util.module_from_spec(spec);sys.modules[spec.name]=M;spec.loader.exec_module(M)
X=M.X;P=M.P;Q=M.Q
recovery=X.load('corrected_budget52','51_budget_recovery.py');recovery.install(X)
C=X.load('corrected_predicate52','52_query_predicate_correction.py');C.install(X)
PATH=P.F.STAGES[52];PATH.mkdir(exist_ok=True);P.F.initialize(52)
LEGACY=C.LEGACY
assert P.F.read_json(P.F.STAGES[51]/'completion_status.json')['stage52_allowed']
queries=P.F.read_json(P.F.STAGES[50]/'questions.json')['queries']
original_plans=P.F.read_json(P.F.STAGES[51]/'repeat1_plans.json')
original_by_query={r['parser']['query_id']:r for r in original_plans['records']}
assert len(original_by_query)==40 and all(r['parser']['repeat']==1 for r in original_by_query.values())
probe=P.F.read_json(LEGACY/'predicate_difference_probe.json');review=P.F.read_json(LEGACY/'predicate_difference_review.json')
assert review['fully_classified_pairs']==len(probe['changed_pairs'])==187 and review['uncertain_or_entire_requested_target_exclusion_pairs']==0
assert review['source_probe_sha256']==P.sha((LEGACY/'predicate_difference_probe.json').read_bytes())
rerouted_path=PATH/'rerouted_repeat1_plans.json'
if rerouted_path.exists():plans=P.F.read_json(rerouted_path)
else:
 records=[]
 for q in queries:
  old_record=original_by_query[q['query_id']];C.guard_installed(X)
  assert Q.original_scope(q['question'])==old_record['original_scope']
  route=Q.route(old_record['parser']['plan'],old_record['original_scope'],parser_failed=old_record['parser']['status']!='ok')
  records.append(dict(parser=old_record['parser'],original_scope=old_record['original_scope'],routing=route,
   correction_provenance=dict(original_record_sha256=P.sha(old_record),predicate_source_sha256=P.sha((ROOT/'notebooks/scripts/52_query_predicate_correction.py').read_bytes()))))
 plans=dict(records=records,original_plans_sha256=P.sha(original_plans),predicate_version=C.VERSION)
 X.frozen(rerouted_path,plans)
assert plans['original_plans_sha256']==P.sha(original_plans)
by_query={r['parser']['query_id']:r for r in plans['records']}
for qid,r in by_query.items():
 assert r['parser']==original_by_query[qid]['parser'] and r['original_scope']==original_by_query[qid]['original_scope']
 assert r['correction_provenance']['predicate_source_sha256']==P.sha((ROOT/'notebooks/scripts/52_query_predicate_correction.py').read_bytes())
annotations=P.F.read_json(LEGACY/'evaluation_annotations.json')
X.frozen(PATH/'frozen_annotations.json',annotations)
old_manifest=P.F.read_json(LEGACY/'execution_manifest.json')
legacy_u={p.stem:p for p in sorted((LEGACY/'retrieval').glob('*:U:*.json'))}
upstream_U_inventory={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in legacy_u.values()}
# No API raw record is copied: original embeddings are accounted exactly once at stage520.
texts=[q['question'] for q in queries];vector_proof=[];loaded=[]
directories=[LEGACY/'api_cache/embeddings']+[P.F.STAGES[s]/'api_cache/embeddings' for s in [47,48,49]]+[P.DATA/d/'api_cache/embeddings' for d in ['42_expanded_query_evaluation','41_query_expansion','40_new_query_comparison','30_structured_benefit_db']]
for text in texts:
 key=P.sha(dict(text=text,model='text-embedding-3-small',dimension=1536))
 source=next((d/(key+'.npy') for d in directories if (d/(key+'.npy')).exists()),None)
 if source is None:raise RuntimeError('Frozen question vector missing; network embedding forbidden')
 loaded.append(np.load(source,allow_pickle=False));vector_proof.append(dict(path=str(source.relative_to(ROOT)),sha256=P.sha(source.read_bytes())))
vectors=np.asarray(loaded,dtype=np.float32)
assert vectors.shape==(40,1536) and np.isfinite(vectors).all()
vector_contract=P.F.read_json(LEGACY/'question_embedding_contract.json')
assert vector_contract==dict(texts_sha256=P.sha(texts),model='text-embedding-3-small',dimensions=1536,matrix_sha256=P.sha(vectors.tobytes()))
X.frozen(PATH/'question_embedding_contract.json',vector_contract)
X.frozen(PATH/'embedding_reuse_provenance.json',dict(original_contract_sha256=P.sha(vector_contract),vector_files=vector_proof,new_requests=0))
qvectors=dict(zip([q['query_id'] for q in queries],vectors,strict=True))
# Cached pair scores are unchanged whole-text BGE outputs, not new measurements.
score_inventory={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in sorted((LEGACY/'bge_cache').glob('*.json'))}
for name,wanted in score_inventory.items():
 source=ROOT/name;target=PATH/'bge_cache'/source.name;target.parent.mkdir(exist_ok=True)
 if target.exists():assert P.sha(target.read_bytes())==wanted
 else:shutil.copyfile(source,target)
identities=[f"{q['query_id']}:S_ONLY:r1" for q in queries]+[f"{q['query_id']}:{pf}:{mode}:r1" for q in queries for pf in X.PROFILES for mode in ['U','SEQ','PAR']]
conditional=[f"{q['query_id']}:{pf}:{mode}:r1" for q in queries for pf in X.PROFILES for mode in ['LLM_SELECTOR','PAR_LLM']]
previous=P.F.read_json(P.F.STAGES[51]/'execution_manifest.json')
sources={**previous['source_sha256'],**{str((ROOT/'notebooks/scripts'/name).relative_to(ROOT)):P.sha((ROOT/'notebooks/scripts'/name).read_bytes()) for name in ['52_retrieval_modes.py','51_budget_recovery.py','52_query_predicate_correction.py']}}
source_inputs={'data/raw/'+d['document_id']+'.pdf':d['source_pdf_sha256'] for d in Q.documents()}
source_inputs.update({str((P.DATA/'30_structured_benefit_db'/name).relative_to(ROOT)):P.sha((P.DATA/'30_structured_benefit_db'/name).read_bytes()) for name in ['structured.sqlite','documents.jsonl','facts.jsonl']})
for name,wanted in source_inputs.items():assert P.sha((ROOT/name).read_bytes())==wanted
manifest=X.make_manifest(52,ROOT/'notebooks/52_corrected_four_retrieval_modes.ipynb',queries,dict(parser=[],answer=identities+conditional),
 maximum_answer=880,maximum_embedding=0,stage_cap=str(C.remaining_stage_cap(X)),extra=dict(source_sha256=sources,split='development',
 logical_answer_conditions=280,conditional_selector_conditions=80,conditional_answer_conditions=80,
 repeat1_plans_sha256=P.sha(plans),original_repeat1_plans_sha256=P.sha(original_plans),annotations_sha256=P.sha(annotations),
 data_sha256={**previous['data_sha256'],**{str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in [LEGACY/'predicate_difference_probe.json',LEGACY/'predicate_difference_review.json',LEGACY/'execution_manifest.json',P.F.STAGES[51]/'repeat1_plans.json',PATH/'rerouted_repeat1_plans.json',PATH/'embedding_reuse_provenance.json']},**upstream_U_inventory,**{r['path']:r['sha256'] for r in vector_proof}},
 source_input_sha256=source_inputs,predicate_version=C.VERSION,source_probe_sha256=P.sha((LEGACY/"predicate_difference_probe.json").read_bytes()),source_probe_review_sha256=P.sha(review),
 upstream_manifest_sha256=P.sha(old_manifest),upstream_U_inventory=upstream_U_inventory,
 embedding_reuse_provenance_sha256=P.sha(P.F.read_json(PATH/'embedding_reuse_provenance.json')),reused_score_inventory=score_inventory,
 original_api_inventory=C.legacy_inventory(X),original_stage52_cost_usd=str(C.legacy_cost_usd(X)),shared_stage52_cap_usd='2',
 pool='all matching DB cards + independent U candidates; no SQL first20 cut',common_unit='whole serialized fact or unchanged operating U evidence chunk',
 card_score='max unit score; tie by card key; disclose unit counts',
 conditional_trigger='source-adjudicated valid card in fixed union but no valid card in BGE top3, ranking cause only'))
X.frozen(PATH/'questions.json',dict(queries=queries,split='development'))
X.frozen(PATH/'evaluation_contract.json',P.F.read_json(LEGACY/'evaluation_contract.json'))
ALLOW_NETWORK=os.getenv('PAID_RETEST_ALLOW_NETWORK')=='1'
print('조회 수정 버전',C.VERSION,'280조건. 기존 검색 임베딩 재사용·신규 임베딩0. 누적비용',X.ledger())


조회 수정 버전 exception-context-v2 280조건. 기존 검색 임베딩 재사용·신규 임베딩0. 누적비용 {'record_count': 441, 'known_priced_records': 440, 'unknown_cost_records': 1, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.272201310', 'tokens': {'input_tokens': 1867285, 'output_tokens': 301488, 'ordinary_input_tokens': 121566, 'cached_input_tokens': 948759, 'cache_write_tokens': 796960, 'embedding_tokens': 9656}, 'billed_cost_usd': None}


In [ ]:
runtime=X.Runtime(52)
started=time.perf_counter();cpu_started=time.process_time()
runtime.last_embedding_seconds=0.0
qvectors=dict(zip([q['query_id'] for q in queries],vectors,strict=True))
for pf in X.PROFILES:runtime.profile(pf)
retrievals=[];search_cache={}
for qi,q in enumerate(queries):
 record=by_query[q['query_id']];route=record['routing'];original=record['original_scope']
 identity=f"{q['query_id']}:S_ONLY:r1";target=PATH/'retrieval'/(identity+'.json')
 new=not target.exists()
 if not new: row=P.F.read_json(target)
 else:
  result=M.rank_units(runtime,q['question'],M.structured_units(route))
  if original['clarify']:result['clarify']=True
  M.assert_scope(result,original['card_keys'])
  row=dict(identity=identity,query_id=q['query_id'],profile='common',mode='S_ONLY',repeat=1,result=result,route=route,
   manifest_sha256=P.sha(manifest),original_scope=original,reuse=None)
 M.bind_retrieval(52,row,question=q,profile='common',mode='S_ONLY',parser_record=record,manifest=manifest,new=new)
 if new:P.F.write_json(target,row)
 retrievals.append(row)
 for pf in X.PROFILES:
  scopes={'U':original['card_keys'],'SEQ':route['scope']}
  results={}
  for mode in ['U','SEQ']:
   identity=f"{q['query_id']}:{pf}:{mode}:r1";target=PATH/'retrieval'/(identity+'.json')
   new=not target.exists()
   upstream=legacy_u.get(identity) if mode=='U' else None
   binding_extra=None
   if upstream is not None:
    old_row=P.F.read_json(upstream)
    M.bind_retrieval(520,old_row,question=q,profile=pf,mode='U',parser_record=original_by_query[q['query_id']],manifest=old_manifest)
    binding_extra=dict(upstream_manifest_sha256=P.sha(old_manifest),upstream_row_sha256=P.sha(old_row),upstream_result_sha256=P.sha(old_row['result']),question_sha256=P.sha(q['question']),profile=pf,original_scope_sha256=P.sha(original),index_data_sha256=old_manifest['data_sha256'],parameters=old_manifest['parameters'],question_vector_sha256=P.sha(qvectors[q['query_id']].tobytes()))
   if not new:row=P.F.read_json(target)
   else:
    key=P.sha(dict(question=q['question'],profile=pf,scope=scopes[mode]));start=time.perf_counter();reuse=None
    if upstream is not None:result=copy.deepcopy(old_row['result']);reuse='verified original U body; historical first-write timing, not a new online measurement'
    elif original['clarify']:result=dict(cards=[],evidence=[],trace={},clarify=True)
    elif key in search_cache:result=copy.deepcopy(search_cache[key]);reuse='same question/profile/scope; original search timing retained'
    else:result=runtime.search(q['question'],qvectors[q['query_id']],pf,scopes[mode])
    M.assert_scope(result,scopes[mode]);M.assert_scope(result,original['card_keys'])
    row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,repeat=1,result=result,route=route if mode=='SEQ' else None,
     manifest_sha256=P.sha(manifest),original_scope=original,reuse=reuse,observed_retrieval_wall_seconds=time.perf_counter()-start,binding_extra_inputs=binding_extra)
   M.bind_retrieval(52,row,question=q,profile=pf,mode=mode,parser_record=record,manifest=manifest,extra_inputs=binding_extra,new=new)
   if new:P.F.write_json(target,row)
   results[mode]=row['result'];search_cache[P.sha(dict(question=q['question'],profile=pf,scope=scopes[mode]))]=row['result'];retrievals.append(row)
  identity=f"{q['query_id']}:{pf}:PAR:r1";target=PATH/'retrieval'/(identity+'.json')
  new=not target.exists()
  if not new:row=P.F.read_json(target)
  else:
   pool=[] if original['clarify'] else M.union_units(M.structured_units(route),M.unstructured_units(runtime,results['U'],pf))
   result=M.rank_units(runtime,q['question'],pool)
   if original['clarify']:result['clarify']=True
   M.assert_scope(result,original['card_keys'])
   row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode='PAR',repeat=1,result=result,route=route,
    manifest_sha256=P.sha(manifest),original_scope=original,reuse=None,independent_U_sha256=P.sha(results['U']))
  M.bind_retrieval(52,row,question=q,profile=pf,mode='PAR',parser_record=record,manifest=manifest,independent_u=results['U'],new=new)
  if new:P.F.write_json(target,row)
  retrievals.append(row)
 for r in retrievals[-7:]:assert r['manifest_sha256']==P.sha(manifest)
 P.F.write_json(PATH/'progress.json',dict(phase='retrieval',questions_completed=qi+1,logical_conditions=len(retrievals),ledger=X.ledger()))
 print('검색',q['query_id'],len(retrievals),flush=True)
retrieval_completed=dict(records=retrievals,resources=runtime.resources(),embedding_wall_seconds=0.0,embedding_note='same frozen vectors reused, no new API or online embedding latency')
retrieval_path=PATH/'retrieval_results.json'
if retrieval_path.exists():assert P.F.read_json(retrieval_path)['records']==retrievals
else:P.F.write_json(retrieval_path,retrieval_completed)
print('검색 완료',len(retrievals),'API 답변 생성 전')

검색 H01 7
검색 H02 14
검색 H03 21
검색 H04 28
`torch_dtype` is deprecated! Use `dtype` instead!
검색 H05 35
검색 H06 42
검색 H07 49
검색 H08 56
검색 H09 63
검색 H10 70


In [ ]:
answers=[]
existing_answer_ids={p.stem for p in (PATH/"answers").glob("*.json")}
initial_api_keys={p.stem for p in (PATH/"api_cache").glob("*.json")}
# 질문별 방식 순서를 번갈아 배치한다. 검색은 공유해도 답변은 mode별 새 ID다.
ordered=sorted(retrievals,key=lambda r:(r['query_id'],(r['mode'] if int(r['query_id'][1:])%2 else ''.join(chr(255-ord(c)) for c in r['mode'])),r['profile']))
for index,row in enumerate(ordered):
 q=next(q for q in queries if q['query_id']==row['query_id'])
 base_u=next((r['result'] for r in retrievals if r['query_id']==row['query_id'] and r['profile']==row['profile'] and r['mode']=='U'),None)
 M.bind_retrieval(52,row,question=q,profile=row['profile'],mode=row['mode'],parser_record=by_query[q['query_id']],manifest=manifest,independent_u=base_u if row['mode']=='PAR' else None,extra_inputs=row.get('binding_extra_inputs'))
 answer=M.answer(52,row['identity'],q['question'],row['result'],allow_network=ALLOW_NETWORK)
 answers.append({**answer,**{k:row[k] for k in ['query_id','profile','mode','repeat']}})
 P.F.write_json(PATH/'progress.json',dict(phase='answer',completed=len(answers),expected=280,last_identity=row['identity'],ledger=X.ledger()))
 print('답변',row['identity'],answer['status'],round(answer['answer_ready_wall_seconds'],3),'초',flush=True)
segment=dict(resources=runtime.resources(),wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu_started,
 existing_answer_count_at_start=len(existing_answer_ids),newly_created_answer_count=len({a['identity'] for a in answers}-existing_answer_ids),
 new_physical_requests=len({p.stem for p in (PATH/'api_cache').glob('*.json')}-initial_api_keys),time_scope='current execution segment; never an online end-to-end measurement')
P.F.write_json(PATH/'latest_completed_segment.json',segment)
completed=dict(answers=answers,retrievals=retrievals,manifest_sha256=P.sha(manifest),first_completed_segment=segment)
result_path=PATH/'execution_results.json'
if result_path.exists():
 saved=P.F.read_json(result_path);assert saved['answers']==answers and saved['retrievals']==retrievals and saved['manifest_sha256']==P.sha(manifest)
else:P.F.write_json(result_path,completed)
P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_source_evaluation',logical_answers=len(answers),expected=280,stage53_allowed=False))
print('280조건 완료. 원문 평가·조건부 LLM 결합 진입 판정 전 최종 선정하지 않는다.',X.ledger())

In [ ]:
# 결과 원문 검토에서 순위 원인이 확인된 경우에만 추가 LLM 선택 비교를 실행한다.
from typing import Literal
from pydantic import create_model, Field, ValidationError

def select_existing_cards(q, profile, pool_result):
 identity=f"{q['query_id']}:{profile}:LLM_SELECTOR:r1";target=PATH/'selectors'/(identity+'.json')
 units=pool_result['trace']['ranked_units'];keys=sorted({u['card_key'] for u in units})
 if not keys:
  return dict(identity=identity,status='empty_pool',selection=[],events=[],wall_seconds=0.)
 schema=create_model('ExistingCardSelection'+str(len(keys)),cards=(list[Literal.__getitem__(tuple(keys))],Field(min_length=0,max_length=3)))
 content=P.F.canonical(dict(question=q['question'],candidate_units=[{k:u[k] for k in ['unit_id','card_key','card_name','origin','text']} for u in units]))
 request_input=[dict(role='user',content=content)]
 contract=dict(execution_manifest_sha256=P.sha(manifest),input_sha256_by_identity={identity:P.sha(request_input)},schema_sha256_by_identity={identity:P.sha(schema.model_json_schema())})
 X.frozen(PATH/'selector_contracts'/(identity+'.json'),contract)
 global_path=PATH/'answer_payload_contract.json';combined=P.F.read_json(global_path)
 for name in ['input_sha256_by_identity','schema_sha256_by_identity']:
  if identity in combined[name]:assert combined[name][identity]==contract[name][identity]
  combined[name].update(contract[name])
 P.F.write_json(global_path,combined)
 api=X.Cache(52,'answer',identity,allow_network=ALLOW_NETWORK)
 instructions='원질문과 후보 근거만 읽고 질문에 실제로 맞는 카드 ID를 최대3개 선택하세요. 후보 밖 카드는 만들지 마세요. 조건을 만족하는 카드가 없으면 빈 목록입니다. 추가 검색은 수행하지 않습니다.'
 if target.exists():row=P.F.read_json(target)
 else:
  start=time.perf_counter();selection=None;error=None
  for attempt in [1,2]:
   kwargs=dict(model='gpt-6-luna',instructions=instructions if attempt==1 else instructions+' 이전 출력의 형식 오류를 고쳐 완전한 JSON으로 답하세요.',input=request_input,text_format=schema,tools=[],store=False,max_output_tokens=2400)
   try:
    response=api.parse(**kwargs);selection=list(response.output_parsed.cards) if response.output_parsed else None
    if selection is None or len(set(selection))!=len(selection):raise ValueError('Empty or duplicate selector output')
    break
   except (ValidationError,ValueError) as exc:error=str(exc)
   except Exception as exc:
    if not api.events:raise
    error=f'{type(exc).__name__}: {exc}';break
  row=dict(identity=identity,status='ok' if selection is not None else 'failed',selection=selection or [],error=error,events=api.events,
   input_sha256=P.sha(request_input),schema_sha256=P.sha(schema.model_json_schema()),manifest_sha256=P.sha(manifest),fixed_pool_sha256=P.sha(pool_result),wall_seconds=time.perf_counter()-start)
  P.F.write_json(target,row)
 assert row['input_sha256']==P.sha(request_input) and row['schema_sha256']==P.sha(schema.model_json_schema()) and row['manifest_sha256']==P.sha(manifest) and row['fixed_pool_sha256']==P.sha(pool_result)
 assert set(row['selection'])<=set(keys)
 for event in row['events']:
  raw=P.F.read_json(PATH/'api_cache'/(event['key']+'.json'));request=raw['request']
  kwargs={k:v for k,v in request.items() if k!='schema'};kwargs['text_format']=schema
  assert api.key(kwargs)==event['key'] and P.sha(request['input'])==row['input_sha256'] and P.sha(request['schema'])==row['schema_sha256']
  assert request['model']=='gpt-6-luna' and request['max_output_tokens']==2400 and request['reasoning']=={'effort':'medium'}
 if row['status']=='ok':
  assert row['events'] and P.F.read_json(PATH/'api_cache'/(row['events'][-1]['key']+'.json'))['parsed']['cards']==row['selection']
 return row

gate_path=PATH/'conditional_trigger_review.json'
if not gate_path.exists():
 print('조건부 LLM 비교는 원문 평가의 진입 판정 대기 중이다. 추가 API 호출 없음.')
else:
 gate=P.F.read_json(gate_path)
 assert gate['manifest_sha256']==P.sha(manifest) and gate['execution_results_sha256']==P.sha(P.F.read_json(PATH/'execution_results.json'))
 review=P.F.read_json(PATH/'source_review.json')
 assert gate['source_review_sha256']==P.sha(review)
 assert review['execution_results_sha256']==gate['execution_results_sha256']
 assert gate['annotations_sha256']==review['annotations_sha256']==manifest['annotations_sha256']==P.sha(P.F.read_json(PATH/'frozen_annotations.json'))
 source_inputs={str((P.DATA/'30_structured_benefit_db'/name).relative_to(ROOT)):P.sha((P.DATA/'30_structured_benefit_db'/name).read_bytes()) for name in ['structured.sqlite','documents.jsonl','facts.jsonl']}
 source_inputs.update({'data/raw/'+d['document_id']+'.pdf':d['source_pdf_sha256'] for d in Q.documents()})
 assert review['source_input_sha256']==gate['source_input_sha256']==source_inputs
 for name,expected in source_inputs.items():assert P.sha((ROOT/name).read_bytes())==expected
 reviewed=review['query_profile_results']
 assert len(reviewed)==80 and {(r['query_id'],r['profile']) for r in reviewed}=={(q['query_id'],pf) for q in queries for pf in X.PROFILES}
 confirmed=set()
 for checked in reviewed:
  q=next(q for q in queries if q['query_id']==checked['query_id'])
  assert checked['question_sha256']==P.sha(q['question'])
  source=next(r for r in retrievals if r['query_id']==q['query_id'] and r['profile']==checked['profile'] and r['mode']=='PAR')
  pool=set(source['result']['trace']['candidate_cards']);top={c['card_key'] for c in source['result']['cards']}
  assert checked['PAR_result_sha256']==P.sha(source['result'])
  judgments={c['card_key']:c for c in checked['card_judgments']}
  assert top<=set(judgments)
  valid={k for k,c in judgments.items() if c['valid_for_original_question'] and k in pool}
  for key,c in judgments.items():
   assert c['reason'] and c['source_support']
   for support in c['source_support']:
    assert support['path']=='data/raw/'+key+'.pdf' and source_inputs[support['path']]==support['sha256'] and support['page']>=1 and support['observation']
  if checked['ranking_cause_confirmed']:
   assert checked['failure_cause']=='ranking' and valid and not (top&valid)
   assert all(not judgments[key]['valid_for_original_question'] for key in top)
   confirmed.add(checked['query_id'])
 assert set(gate['source_adjudicated_ranking_failure_query_ids'])==confirmed
 assert gate['trigger']==bool(confirmed)
 if not gate['trigger']:
  print('BGE 순위 원인 진입 조건 미충족: 조건부 LLM 호출 없음.')
 else:
  assert gate['source_adjudicated_ranking_failure_query_ids'] and gate['all_cases_evaluated']
  selectors=[];alternative=[]
  for q in queries:
   for pf in X.PROFILES:
    bge=next(r for r in retrievals if r['query_id']==q['query_id'] and r['profile']==pf and r['mode']=='PAR')
    u=next(r['result'] for r in retrievals if r['query_id']==q['query_id'] and r['profile']==pf and r['mode']=='U')
    M.bind_retrieval(52,bge,question=q,profile=pf,mode='PAR',parser_record=by_query[q['query_id']],manifest=manifest,independent_u=u)
    selector=select_existing_cards(q,pf,bge['result']);selectors.append(selector)
    kept=[unit for unit in bge['result']['trace']['ranked_units'] if unit['card_key'] in selector['selection']]
    result=M.pack_ranked(q['question'],kept,[unit['score'] for unit in kept])
    if bge['original_scope']['clarify']:result['clarify']=True
    M.assert_scope(result,bge['original_scope']['card_keys'])
    identity=f"{q['query_id']}:{pf}:PAR_LLM:r1"
    a=M.answer(52,identity,q['question'],result,allow_network=ALLOW_NETWORK)
    alternative.append(dict(answer=a,query_id=q['query_id'],profile=pf,mode='PAR_LLM',repeat=1,result=result,selector_sha256=P.sha(selector),selector_status=selector['status']))
    P.F.write_json(PATH/'conditional_progress.json',dict(completed=len(alternative),expected=80,ledger=X.ledger()))
  P.F.write_json(PATH/'conditional_execution_results.json',dict(selectors=selectors,answers=alternative,decision_pending_source_evaluation=True,
   card_choice='LLM subset of same fixed pool; BGE order retained among chosen cards/evidence',ledger=X.ledger()))
  print('조건부80문항·프로필 실행완료. BGE와 원문 품질 및 비용 비교 뒤 선정한다.')
